# FarmTech Solutions — Fase 6
## Detecção de objetos com YOLOv5

**FIAP — Curso de Inteligência Artificial**

**Grupo:**
- Karina Garta Szewczuk — RM569309
- Maria Sabrina Feitosa da Silva — RM568714
- Nicolas Lima Apolinário — RM570741
- Roger Gabriel de Souza Jesus Costa — RM573659

---

Nesse notebook a gente treina a YOLOv5 pra detectar dois objetos que escolhemos pro projeto. A base tem 80 imagens no total (40 de cada), rotuladas no Make Sense. O treino roda duas vezes, uma com 30 épocas e outra com 60, pra comparar se treinar mais tempo faz diferença.

**Importante antes de rodar:**
- Ligar a GPU no Colab (Ambiente de execução → Alterar tipo → T4 GPU)
- Ter as imagens organizadas no Drive conforme a estrutura abaixo
- Conferir se a ordem das classes no `data.yaml` bate com a ordem que criou no Make Sense

Estrutura esperada no Drive:
```
MyDrive/farmtech_fase6/
└── dataset/
    ├── images/
    │   ├── train/   (64 imagens)
    │   ├── val/     (8 imagens)
    │   └── test/    (8 imagens)
    └── labels/
        ├── train/   (64 .txt do Make Sense)
        └── val/     (8 .txt do Make Sense)
```
As imagens de teste não precisam de rótulo, só as de treino e validação.


## 1. Montar o Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configuração

Ajusta aqui o caminho da base e o nome das classes. Os nomes precisam estar na mesma ordem do Make Sense (a primeira classe que criou lá é a classe 0).

In [ ]:
# === AJUSTAR AQUI ===
BASE = '/content/drive/MyDrive/farmtech_fase6/dataset'

CLASSE_A = 'objeto_a'   # trocar pelo nome do primeiro objeto
CLASSE_B = 'objeto_b'   # trocar pelo nome do segundo objeto

IMG_SIZE = 640
BATCH    = 16
EPOCAS_1 = 30
EPOCAS_2 = 60

print('Base:', BASE)
print('Classes:', [CLASSE_A, CLASSE_B])


## 3. Clonar a YOLOv5

In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -qr requirements.txt


In [ ]:
import torch
print('Torch:', torch.__version__)
print('GPU:', 'cuda' if torch.cuda.is_available() else 'sem GPU!')
!nvidia-smi


## 4. Montar o data.yaml

Esse arquivo diz pra YOLOv5 onde estão as imagens e quais são as classes.

In [ ]:
import yaml

config = {
    'train': f'{BASE}/images/train',
    'val':   f'{BASE}/images/val',
    'nc':    2,
    'names': [CLASSE_A, CLASSE_B],
}

with open('data.yaml', 'w') as f:
    yaml.dump(config, f, sort_keys=False)

print(open('data.yaml').read())


In [ ]:
import glob

# confere se ta tudo certo antes de treinar
for sub in ['images/train', 'images/val', 'images/test', 'labels/train', 'labels/val']:
    qtd = len(glob.glob(f'{BASE}/{sub}/*'))
    print(f'{sub:15s}: {qtd} arquivos')


## 5. Treino com 30 épocas

Parte dos pesos pré-treinados da `yolov5s.pt` (transfer learning).

In [ ]:
!python train.py \
  --img {IMG_SIZE} \
  --batch {BATCH} \
  --epochs {EPOCAS_1} \
  --data data.yaml \
  --weights yolov5s.pt \
  --name treino_30ep \
  --cache


## 6. Treino com 60 épocas

Mesma coisa, só dobrando as épocas pra ver se melhora.

In [ ]:
!python train.py \
  --img {IMG_SIZE} \
  --batch {BATCH} \
  --epochs {EPOCAS_2} \
  --data data.yaml \
  --weights yolov5s.pt \
  --name treino_60ep \
  --cache


## 7. Comparação 30 vs 60 épocas

Lendo o results.csv de cada treino pra comparar as métricas finais e plotar as curvas.

In [ ]:
import pandas as pd

res_30 = pd.read_csv('runs/train/treino_30ep/results.csv')
res_60 = pd.read_csv('runs/train/treino_60ep/results.csv')

# limpa espaços nos nomes das colunas
res_30.columns = res_30.columns.str.strip()
res_60.columns = res_60.columns.str.strip()

final = pd.DataFrame({
    '30 épocas': res_30.iloc[-1],
    '60 épocas': res_60.iloc[-1],
})

metricas = ['metrics/precision', 'metrics/recall', 'metrics/mAP_0.5', 'metrics/mAP_0.5:0.95']
final.loc[metricas].round(4)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(res_30['epoch'], res_30['metrics/mAP_0.5'], marker='o', label='30 épocas')
axes[0].plot(res_60['epoch'], res_60['metrics/mAP_0.5'], marker='o', label='60 épocas')
axes[0].set_title('mAP@0.5 por época')
axes[0].set_xlabel('época')
axes[0].set_ylabel('mAP@0.5')
axes[0].legend()

axes[1].plot(res_30['epoch'], res_30['val/obj_loss'], marker='o', label='30 épocas')
axes[1].plot(res_60['epoch'], res_60['val/obj_loss'], marker='o', label='60 épocas')
axes[1].set_title('Perda de validação por época')
axes[1].set_xlabel('época')
axes[1].set_ylabel('val/obj_loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/comparacao_30_60.png', dpi=120)
plt.show()


*(preencher depois de rodar)*

Qual treino ficou melhor? O de 60 épocas melhorou em relação ao de 30, ou já tinha estabilizado? A loss de validação continuou caindo ou começou a subir (overfitting)?

## 8. Validação

In [ ]:
# valida o modelo de 30 épocas
!python val.py \
  --weights runs/train/treino_30ep/weights/best.pt \
  --data data.yaml \
  --img {IMG_SIZE} \
  --name val_30ep


In [ ]:
# valida o modelo de 60 épocas
!python val.py \
  --weights runs/train/treino_60ep/weights/best.pt \
  --data data.yaml \
  --img {IMG_SIZE} \
  --name val_60ep


## 9. Detecção nas imagens de teste

Rodando o modelo de 60 épocas nas imagens que ele nunca viu.

In [ ]:
!python detect.py \
  --weights runs/train/treino_60ep/weights/best.pt \
  --source {BASE}/images/test \
  --img {IMG_SIZE} \
  --conf 0.25 \
  --name teste_60ep


In [ ]:
from IPython.display import Image, display
import glob

imgs = glob.glob('runs/detect/teste_60ep/*.jpg') + glob.glob('runs/detect/teste_60ep/*.png')
print(f'{len(imgs)} imagens de teste:\n')
for caminho in sorted(imgs):
    display(Image(filename=caminho, width=500))


## 10. Salvar o modelo no Drive

Salvando o best.pt no Drive pra usar na Entrega 2.

In [ ]:
import shutil, os

RAIZ = os.path.dirname(BASE)

shutil.copy('runs/train/treino_60ep/weights/best.pt', f'{RAIZ}/best.pt')
shutil.copy('/content/comparacao_30_60.png', f'{RAIZ}/comparacao_30_60.png')

print('Salvo:')
print(f'  {RAIZ}/best.pt')
print(f'  {RAIZ}/comparacao_30_60.png')


## 11. Conclusões

*(preencher depois de rodar)*

**30 vs 60 épocas:**
- qual teve melhor mAP? o ganho de 30 pra 60 compensou? teve overfitting?

**Detecções no teste:**
- acertou os dois objetos? errou algum? em que situação errou?

**Pontos fortes:**
- transfer learning com yolov5s.pt, que funciona bem mesmo com poucas imagens
- comparação direta entre duas configs de treino
- pipeline completo: drive → treino → validação → teste

**Limitações:**
- base pequena (80 imagens), pode não generalizar muito
- só duas classes
- resultado depende bastante da qualidade das fotos e da rotulação
